In [21]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Pandas version: 2.0.3
NumPy version: 1.24.4


In [22]:
# Load the cleaned dataset
df = pd.read_csv("../data/hotel_bookings_clean.csv")

print("Dataset Shape:", df.shape)
print("Columns:", df.shape[1])

Dataset Shape: (87396, 32)
Columns: 32


In [23]:
# Check missing values and duplicate rows

print("Duplicate rows:", df.duplicated().sum())

missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

print("\nColumns with missing values:")
print(missing_values)

Duplicate rows: 0

Columns with missing values:
COMPANY     82137
AGENT       12193
COUNTRY       452
CHILDREN        4
dtype: int64


In [24]:
# Basic stay and guest features

df["TOTAL_STAY_NIGHTS"] = (
    df["STAYS_IN_WEEKEND_NIGHTS"]
    + df["STAYS_IN_WEEK_NIGHTS"]
)

df["TOTAL_GUESTS"] = (
    df["ADULTS"]
    + df["CHILDREN"].fillna(0)
    + df["BABIES"]
)

df["TOTAL_SPECIAL_REQUESTS"] = df["TOTAL_OF_SPECIAL_REQUESTS"]

print("Feature engineering completed.")
print(
    df[
        [
            "TOTAL_STAY_NIGHTS",
            "TOTAL_GUESTS",
            "TOTAL_SPECIAL_REQUESTS"
        ]
    ].head()
)

Feature engineering completed.
   TOTAL_STAY_NIGHTS  TOTAL_GUESTS  TOTAL_SPECIAL_REQUESTS
0                  2           3.0                       1
1                  7           2.0                       0
2                  1           2.0                       1
3                  4           2.0                       0
4                  2           2.0                       2


In [25]:
# Estimated room revenue

df["ESTIMATED_REVENUE"] = (
    df["ADR"] * df["TOTAL_STAY_NIGHTS"]
)

print("Estimated revenue feature created.")

print(
    df[
        [
            "ADR",
            "TOTAL_STAY_NIGHTS",
            "ESTIMATED_REVENUE"
        ]
    ].head()
)

Estimated revenue feature created.
      ADR  TOTAL_STAY_NIGHTS  ESTIMATED_REVENUE
0  205.10                  2             410.20
1  104.85                  7             733.95
2  140.00                  1             140.00
3  126.20                  4             504.80
4  200.00                  2             400.00


In [26]:
# Booking history and change-related features

df["HAS_BOOKING_CHANGES"] = (
    df["BOOKING_CHANGES"] > 0
).astype(int)

df["HAS_PREVIOUS_CANCELLATIONS"] = (
    df["PREVIOUS_CANCELLATIONS"] > 0
).astype(int)

df["HAS_PREVIOUS_BOOKINGS"] = (
    df["PREVIOUS_BOOKINGS_NOT_CANCELED"] > 0
).astype(int)

df["HAS_SPECIAL_REQUESTS"] = (
    df["TOTAL_OF_SPECIAL_REQUESTS"] > 0
).astype(int)

print("Behavioral features created.")

print(
    df[
        [
            "BOOKING_CHANGES",
            "HAS_BOOKING_CHANGES",
            "PREVIOUS_CANCELLATIONS",
            "HAS_PREVIOUS_CANCELLATIONS",
            "PREVIOUS_BOOKINGS_NOT_CANCELED",
            "HAS_PREVIOUS_BOOKINGS",
            "TOTAL_OF_SPECIAL_REQUESTS",
            "HAS_SPECIAL_REQUESTS"
        ]
    ].head()
)

Behavioral features created.
   BOOKING_CHANGES  HAS_BOOKING_CHANGES  PREVIOUS_CANCELLATIONS  \
0                0                    0                       0   
1                0                    0                       0   
2                1                    1                       0   
3                0                    0                       0   
4                1                    1                       0   

   HAS_PREVIOUS_CANCELLATIONS  PREVIOUS_BOOKINGS_NOT_CANCELED  \
0                           0                               0   
1                           0                               0   
2                           0                               0   
3                           0                               0   
4                           0                               0   

   HAS_PREVIOUS_BOOKINGS  TOTAL_OF_SPECIAL_REQUESTS  HAS_SPECIAL_REQUESTS  
0                      0                          1                     1  
1                      0 

In [27]:
# Categorize bookings based on lead time

df["LEAD_TIME_CATEGORY"] = pd.cut(
    df["LEAD_TIME"],
    bins=[-1, 7, 30, 90, 180, float("inf")],
    labels=[
        "0-7 Days",
        "8-30 Days",
        "31-90 Days",
        "91-180 Days",
        "180+ Days"
    ]
)

print("Lead time categories created.")

print(
    df["LEAD_TIME_CATEGORY"]
    .value_counts()
    .sort_index()
)

Lead time categories created.
LEAD_TIME_CATEGORY
0-7 Days       18304
8-30 Days      16340
31-90 Days     22744
91-180 Days    18243
180+ Days      11765
Name: count, dtype: int64


In [28]:
# Check the engineered features

engineered_features = [
    "TOTAL_STAY_NIGHTS",
    "TOTAL_GUESTS",
    "TOTAL_SPECIAL_REQUESTS",
    "ESTIMATED_REVENUE",
    "HAS_BOOKING_CHANGES",
    "HAS_PREVIOUS_CANCELLATIONS",
    "HAS_PREVIOUS_BOOKINGS",
    "HAS_SPECIAL_REQUESTS",
    "LEAD_TIME_CATEGORY"
]

print("Engineered features:")
print(engineered_features)

print("\nTotal features in dataset:", df.shape[1])

df[engineered_features].head()

Engineered features:
['TOTAL_STAY_NIGHTS', 'TOTAL_GUESTS', 'TOTAL_SPECIAL_REQUESTS', 'ESTIMATED_REVENUE', 'HAS_BOOKING_CHANGES', 'HAS_PREVIOUS_CANCELLATIONS', 'HAS_PREVIOUS_BOOKINGS', 'HAS_SPECIAL_REQUESTS', 'LEAD_TIME_CATEGORY']

Total features in dataset: 41


,TOTAL_STAY_NIGHTS,TOTAL_GUESTS,TOTAL_SPECIAL_REQUESTS,ESTIMATED_REVENUE,HAS_BOOKING_CHANGES,HAS_PREVIOUS_CANCELLATIONS,HAS_PREVIOUS_BOOKINGS,HAS_SPECIAL_REQUESTS,LEAD_TIME_CATEGORY
0,2,3.0,1,410.20,0,0,0,1,8-30 Days
1,7,2.0,0,733.95,0,0,0,0,91-180 Days
2,1,2.0,1,140.00,1,0,0,1,0-7 Days
3,4,2.0,0,504.80,0,0,0,0,91-180 Days
4,2,2.0,2,400.00,1,0,0,1,91-180 Days


In [29]:
# Define target variable and check potentially problematic columns

TARGET = "IS_CANCELED"

print("Target:", TARGET)
print("Target distribution:")
print(df[TARGET].value_counts())
print("\nTarget percentage:")
print((df[TARGET].value_counts(normalize=True) * 100).round(2))

print("\nPotential leakage / post-booking columns:")
print([
    col for col in [
        "RESERVATION_STATUS",
        "RESERVATION_STATUS_DATE",
        "DAYS_IN_WAITING_LIST",
        "TOTAL_OF_SPECIAL_REQUESTS"
    ]
    if col in df.columns
])

Target: IS_CANCELED
Target distribution:
IS_CANCELED
0    63371
1    24025
Name: count, dtype: int64

Target percentage:
IS_CANCELED
0    72.51
1    27.49
Name: proportion, dtype: float64

Potential leakage / post-booking columns:
['RESERVATION_STATUS', 'RESERVATION_STATUS_DATE', 'DAYS_IN_WAITING_LIST', 'TOTAL_OF_SPECIAL_REQUESTS']


In [30]:
# Remove post-outcome information to prevent data leakage

leakage_columns = [
    "RESERVATION_STATUS",
    "RESERVATION_STATUS_DATE"
]

model_df = df.drop(columns=leakage_columns).copy()

print("Removed leakage columns:", leakage_columns)
print("Model dataset shape:", model_df.shape)

Removed leakage columns: ['RESERVATION_STATUS', 'RESERVATION_STATUS_DATE']
Model dataset shape: (87396, 39)


In [31]:
# Handle missing values for modeling

model_df["COMPANY"] = model_df["COMPANY"].fillna("Unknown")
model_df["AGENT"] = model_df["AGENT"].fillna("Unknown")
model_df["COUNTRY"] = model_df["COUNTRY"].fillna("Unknown")
model_df["CHILDREN"] = model_df["CHILDREN"].fillna(0)

print("Missing values after treatment:")

remaining_missing = model_df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

print(remaining_missing)

print("\nModel dataset shape:", model_df.shape)

Missing values after treatment:
Series([], dtype: int64)

Model dataset shape: (87396, 39)


In [32]:
# Identify categorical columns

categorical_cols = model_df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Categorical columns:")
for col in categorical_cols:
    print("-", col)

print("\nNumber of categorical columns:", len(categorical_cols))

Categorical columns:
- HOTEL
- ARRIVAL_DATE_MONTH
- MEAL
- COUNTRY
- MARKET_SEGMENT
- DISTRIBUTION_CHANNEL
- RESERVED_ROOM_TYPE
- ASSIGNED_ROOM_TYPE
- DEPOSIT_TYPE
- AGENT
- COMPANY
- CUSTOMER_TYPE
- LEAD_TIME_CATEGORY

Number of categorical columns: 13


In [33]:
# Check cardinality of categorical features

for col in categorical_cols:
    print(f"{col}: {model_df[col].nunique()} unique values")

HOTEL: 2 unique values
ARRIVAL_DATE_MONTH: 12 unique values
MEAL: 5 unique values
COUNTRY: 178 unique values
MARKET_SEGMENT: 8 unique values
DISTRIBUTION_CHANNEL: 5 unique values
RESERVED_ROOM_TYPE: 10 unique values
ASSIGNED_ROOM_TYPE: 12 unique values
DEPOSIT_TYPE: 3 unique values
AGENT: 334 unique values
COMPANY: 353 unique values
CUSTOMER_TYPE: 4 unique values
LEAD_TIME_CATEGORY: 5 unique values


In [34]:
# Select categorical columns for one-hot encoding

encoding_cols = [
    "HOTEL",
    "ARRIVAL_DATE_MONTH",
    "MEAL",
    "MARKET_SEGMENT",
    "DISTRIBUTION_CHANNEL",
    "RESERVED_ROOM_TYPE",
    "ASSIGNED_ROOM_TYPE",
    "DEPOSIT_TYPE",
    "CUSTOMER_TYPE",
    "LEAD_TIME_CATEGORY"
]

high_cardinality_cols = [
    "COUNTRY",
    "AGENT",
    "COMPANY"
]

# Remove high-cardinality columns
model_df = model_df.drop(columns=high_cardinality_cols)

# One-hot encode selected categorical features
model_df_encoded = pd.get_dummies(
    model_df,
    columns=encoding_cols,
    drop_first=True,
    dtype=int
)

print("High-cardinality columns removed:", high_cardinality_cols)
print("Encoded dataset shape:", model_df_encoded.shape)

High-cardinality columns removed: ['COUNTRY', 'AGENT', 'COMPANY']
Encoded dataset shape: (87396, 82)


In [44]:
print([
    col for col in model_df_encoded.columns
    if col.startswith("CUSTOMER_TYPE_")
])

['CUSTOMER_TYPE_Group', 'CUSTOMER_TYPE_Transient', 'CUSTOMER_TYPE_Transient-Party']


In [35]:
# Separate target variable from input features

X = model_df_encoded.drop(columns=["IS_CANCELED"])
y = model_df_encoded["IS_CANCELED"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (87396, 81)
Target shape: (87396,)


In [36]:
pip install scikit-learn

You should consider upgrading via the 'd:\TravelIQ\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [37]:
# Strict time-based split by complete months

# Create a sortable date key
model_df_encoded['MONTH_NUM'] = df['ARRIVAL_DATE_MONTH'].map(month_map)
model_df_encoded['YEAR'] = df['ARRIVAL_DATE_YEAR']

# Sort chronologically
model_df_sorted = model_df_encoded.sort_values(
    ['YEAR', 'MONTH_NUM']
).reset_index(drop=True)

# Get unique year-month periods
periods = (
    model_df_sorted[['YEAR', 'MONTH_NUM']]
    .drop_duplicates()
    .sort_values(['YEAR', 'MONTH_NUM'])
    .reset_index(drop=True)
)

# Use first 80% of complete months for training
split_period_idx = int(len(periods) * 0.80)

train_periods = periods.iloc[:split_period_idx]
test_periods = periods.iloc[split_period_idx:]

# Create masks
train_mask = model_df_sorted.set_index(
    ['YEAR', 'MONTH_NUM']
).index.isin(
    [tuple(x) for x in train_periods.values]
)

test_mask = model_df_sorted.set_index(
    ['YEAR', 'MONTH_NUM']
).index.isin(
    [tuple(x) for x in test_periods.values]
)

train_df = model_df_sorted[train_mask]
test_df = model_df_sorted[test_mask]

# Separate features and target
X_train = train_df.drop(columns=['IS_CANCELED', 'YEAR', 'MONTH_NUM'])
y_train = train_df['IS_CANCELED']

X_test = test_df.drop(columns=['IS_CANCELED', 'YEAR', 'MONTH_NUM'])
y_test = test_df['IS_CANCELED']

print("Train:", X_train.shape)
print("Test:", X_test.shape)

print(
    "Train period:",
    train_periods.iloc[0].tolist(),
    "to",
    train_periods.iloc[-1].tolist()
)

print(
    "Test period:",
    test_periods.iloc[0].tolist(),
    "to",
    test_periods.iloc[-1].tolist()
)

NameError: name 'month_map' is not defined

In [39]:
# Save final feature-engineered dataset

model_df_encoded.to_csv(
    "../data/model_ready_data.csv",
    index=False
)

print("Final model-ready dataset saved.")
print("Shape:", model_df_encoded.shape)

Final model-ready dataset saved.
Shape: (87396, 82)


In [43]:
# Save the exact time-based train/test split

train_df.to_csv(
    "../data/train_data.csv",
    index=False
)

test_df.to_csv(
    "../data/test_data.csv",
    index=False
)

print("Train and test datasets saved separately.")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

NameError: name 'train_df' is not defined

In [41]:
print(sorted(model_df["CUSTOMER_TYPE"].dropna().unique()))

['Contract', 'Group', 'Transient', 'Transient-Party']
